# LuNaVision AI: Agentic TESS Exoplanet Platform (Kaggle GPU notebook)

A multi-agent system that takes TESS light curves to vetted planet candidates, with **trainable agents** and an **orchestrator that observes results and decides what to do next**.

| Agent | Job | Trainable / GPU |
|---|---|---|
| `orchestrator` | plans stages, retries, adapts to what it observes, resumes runs | policy rules |
| `acquisition` | inventory of light curves, optional MAST download | |
| `preprocessing` | normalise, clip, bi-weight detrend, quality gate | |
| `detection` | Transit Least Squares scan of new curves | |
| `features` | 65+ features per candidate | **GPU** (CuPy / PyTorch) |
| `classification` | stacked, calibrated ensemble; Optuna search; model registry | **trained on GPU** (XGBoost) |
| `deep_vetting` | 1D CNN on phase-folded global + local views | **trained on GPU** (PyTorch, AMP) |
| `consensus` | combines tabular + CNN opinions, measures disagreement | |
| `active_learning` | which candidates a human should label next; simulated label-efficiency experiment | **learned query policy** |
| `fitting` | batman + emcee posteriors, self-heals non-converged fits | |
| `reporting` | 7 plots, vetting checks, ranked catalogue, 3-page PDFs | |
| `explainer` | plain-English candidate summaries with an open LLM (template fallback) | GPU LLM |
| `monitor` | per-stage timing, GPU and RAM usage | |

**How to run:** run the cells top to bottom. Leave `SMOKE = True` the first time (about 10-15 min) to prove every agent works, then set `SMOKE = False` and rerun from the config cell.

**What to attach (Add Input):** a dataset with the repo `src/` folder and catalogs (`feature_matrix.csv`, `unified_labels.csv`, `synthetic_injections.csv`, ideally `tic_stellar_params.csv`), plus **either** `lc_detrended/` **or** `lc_raw/` (+ `lc_injected/`). The setup cell discovers them automatically.

## 0. Settings

In [ ]:
# ===================== USER SETTINGS =====================
SMOKE   = True     # True = tiny fast run to verify every agent. False = full run.
USE_LLM = True     # explainer agent narrates candidates with an open LLM (needs internet); falls back to templates
LLM_ID  = "Qwen/Qwen2.5-3B-Instruct"
# =========================================================

from dataclasses import dataclass

@dataclass
class Config:
    smoke: bool = True
    seed: int = 42
    n_top: int = 30            # candidates to fit and report
    min_prob: float = 0.5      # consensus probability threshold for candidates
    optuna_trials: int = 40    # XGBoost GPU hyper-parameter search
    cnn_epochs: int = 40
    cnn_max_rows: int = 0      # 0 = all rows
    fit_steps: int = 1000      # emcee minimum steps
    fit_max_steps: int = 4000
    al_rounds: int = 8         # active-learning simulation
    al_batch: int = 25
    al_seeds: int = 3
    use_llm: bool = True
    llm_id: str = "Qwen/Qwen2.5-3B-Instruct"
    llm_top: int = 10          # candidates to narrate

    def apply_smoke(self):
        self.optuna_trials, self.cnn_epochs, self.cnn_max_rows = 4, 2, 400
        self.n_top, self.fit_steps, self.fit_max_steps = 3, 300, 300
        self.al_rounds, self.al_batch, self.al_seeds = 3, 10, 2
        self.use_llm, self.llm_top = False, 3
        return self

CFG = Config(smoke=SMOKE, use_llm=USE_LLM, llm_id=LLM_ID)
if SMOKE:
    CFG.apply_smoke()
print(CFG)

## 1. Environment

In [ ]:
!pip install -q emcee corner transitleastsquares batman-package wotan lightkurve astroquery optuna shap imbalanced-learn "transformers>=4.44" accelerate cupy-cuda12x 2>&1 | tail -2

In [ ]:
import subprocess, os
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout or "no nvidia-smi: GPU accelerator is off (Settings -> Accelerator)")
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
try:
    import cupy as cp
    print("cupy", cp.__version__, "| gpu ok:", float(cp.arange(10).sum()) == 45.0)
except Exception as e:
    print("cupy unavailable, feature extraction falls back to CPU:", e)
import xgboost
print("xgboost", xgboost.__version__)

## 2. Workspace: discover inputs and assemble the project

In [ ]:
import os, sys, shutil, glob, json, time, re, math, random, warnings, traceback, uuid, hashlib, subprocess
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

IN = Path("/kaggle/input")
WS = Path("/kaggle/working/exoplanet_pipeline")
DATA, CAT, MODELS = WS / "data", WS / "data" / "catalogs", WS / "models"
OUT     = WS / ("outputs_smoke" if SMOKE else "outputs")
RUN_ID  = "smoke" if SMOKE else "full"
RUN_DIR = WS / "runs" / RUN_ID
DET, RAW, INJ = DATA / "processed" / "lc_detrended", DATA / "processed" / "lc_raw", DATA / "processed" / "lc_injected"
CATALOG_FILES = ["feature_matrix.csv", "unified_labels.csv", "synthetic_injections.csv", "tic_stellar_params.csv",
                 "preprocessing_report.csv", "exofop_toi_labeled.csv", "nasa_confirmed_labeled.csv", "spoc_tce_sector01.csv"]

def scan_inputs():
    found = {"src": None, "dirs": {}, "files": {}, "model": None}
    for root, dirs, files in os.walk(IN):
        p = Path(root)
        if p.name == "src" and (p / "fitting.py").exists() and found["src"] is None:
            found["src"] = p
        if p.name in ("lc_detrended", "lc_raw", "lc_injected") and p.name not in found["dirs"]:
            found["dirs"][p.name] = p
        if "classifier.joblib" in files and found["model"] is None:
            found["model"] = p / "classifier.joblib"
        for f in files:
            if f in CATALOG_FILES and f not in found["files"]:
                found["files"][f] = p / f
    return found

def dir_size_gb(p):
    return sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file()) / 1e9

found = scan_inputs()
print("src        :", found["src"])
print("light-curve dirs:", {k: str(v) for k, v in found["dirs"].items()})
print("catalogs   :", sorted(found["files"]))
assert found["src"] is not None, "No src/ folder (with fitting.py) found under /kaggle/input. Attach the dataset that contains the repo src/."

# ---- assemble writable workspace ----
for d in (CAT, MODELS, OUT, RUN_DIR, WS / "cache", DATA / "processed"):
    d.mkdir(parents=True, exist_ok=True)
shutil.copytree(found["src"], WS / "src", dirs_exist_ok=True)
for name, p in found["files"].items():
    if not (CAT / name).exists():
        shutil.copy(p, CAT / name)
if found["model"] is not None and not (MODELS / "classifier.joblib").exists():
    shutil.copy(found["model"], MODELS / "classifier.joblib")
for name, dest in (("lc_detrended", DET), ("lc_raw", RAW), ("lc_injected", INJ)):
    if name in found["dirs"] and not dest.exists():
        src_dir = found["dirs"][name]
        if dir_size_gb(src_dir) < 3.5:
            shutil.copytree(src_dir, dest)             # writable copy
        else:
            os.symlink(src_dir, dest)                  # too big to copy; read-only link
        print(f"  attached {name}: {len(list(dest.glob('*.npz')))} light curves")

# ---- idempotent patch of an older report.py (corner plot crashed on zero-width posteriors) ----
rp = WS / "src" / "report.py"
s = rp.read_text()
if "np.ptp(data" not in s and "    fig = corner.corner(data," in s:
    guard = "    if len(data) < 50 or not np.all(np.ptp(data, axis=0) > 0):\n        return None\n"
    rp.write_text(s.replace("    fig = corner.corner(data,", guard + "    fig = corner.corner(data,", 1))
    print("patched report.py (corner guard)")

# joblib worker processes must be able to import the pipeline modules
os.environ["PYTHONPATH"] = f"{WS / 'src'}:" + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, str(WS / "src"))
os.chdir(WS)

import acquisition, preprocessing, detection, features, classifier, fitting, report
print("\nimported pipeline modules OK")

inv = {"lc_detrended": len(list(DET.glob("*.npz"))) if DET.exists() else 0,
       "lc_raw": len(list(RAW.glob("*.npz"))) if RAW.exists() else 0,
       "lc_injected": len(list(INJ.glob("*.npz"))) if INJ.exists() else 0,
       "feature_rows": (sum(1 for _ in open(CAT / "feature_matrix.csv")) - 1) if (CAT / "feature_matrix.csv").exists() else 0}
print("inventory:", inv)
if inv["lc_detrended"] == 0 and inv["lc_raw"] == 0:
    print("WARNING: no light curves attached. Attach lc_detrended/ or lc_raw/ (or use the acquisition agent to download).")

## 3. Agent framework

Agents never call each other directly. Every request is a typed `Message` (`sender, recipient, action, payload, status`) sent through the `MessageBus`, which writes an audit log (`runs/<id>/messages.jsonl`). A shared `Blackboard` keeps run state so a run can be resumed. `BaseAgent.receive` turns any exception into an `error` reply, so one failing agent never crashes the platform: the orchestrator decides what to do.

In [ ]:
from dataclasses import dataclass, field, asdict
from typing import Optional

@dataclass
class Message:
    sender: str
    recipient: str
    action: str
    payload: dict = field(default_factory=dict)
    status: str = "pending"                      # pending | done | error
    run_id: str = ""
    msg_id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])
    reply_to: Optional[str] = None
    error: Optional[str] = None
    ts: float = field(default_factory=time.time)
    seconds: Optional[float] = None

def _short(v, n=200):
    try:
        s = json.dumps(v, default=str)
    except Exception:
        s = repr(v)
    return s if len(s) <= n else s[:n] + "..."

class MessageBus:
    def __init__(self, log_path, verbose=True):
        self.agents, self.log_path, self.verbose = {}, Path(log_path), verbose
        self.log_path.parent.mkdir(parents=True, exist_ok=True)

    def register(self, agent):
        self.agents[agent.name] = agent

    def _log(self, m: Message):
        rec = {"msg_id": m.msg_id, "reply_to": m.reply_to, "ts": round(m.ts, 2), "sender": m.sender,
               "recipient": m.recipient, "action": m.action, "status": m.status,
               "seconds": None if m.seconds is None else round(m.seconds, 2),
               "payload": _short(m.payload), "error": (m.error or "")[:300] or None}
        with open(self.log_path, "a") as f:
            f.write(json.dumps(rec) + "\n")

    def send(self, msg: Message) -> Message:
        msg.run_id = RUN_ID
        self._log(msg)
        if msg.recipient not in self.agents:
            reply = Message(msg.recipient, msg.sender, msg.action + ".result", {}, "error", RUN_ID, reply_to=msg.msg_id,
                            error=f"no agent named {msg.recipient!r}")
        else:
            reply = self.agents[msg.recipient].receive(msg)
        self._log(reply)
        if self.verbose:
            tag = "ok " if reply.status == "done" else "ERR"
            print(f"  [{msg.sender} -> {msg.recipient}] {msg.action}: {tag} ({(reply.seconds or 0):.1f}s)")
            if reply.status == "error":
                print("     " + (reply.error or "").splitlines()[0])
        return reply

class Blackboard:
    """Shared, persisted run state: stage status, facts, and orchestrator decisions."""
    def __init__(self, path):
        self.path = Path(path)
        self.d = json.loads(self.path.read_text()) if self.path.exists() else {"stages": {}, "facts": {}, "decisions": []}

    def save(self):
        self.path.write_text(json.dumps(self.d, indent=1, default=str))

    def mark(self, stage, status, info=None, seconds=None):
        self.d["stages"][stage] = {"status": status, "info": info, "seconds": seconds, "ts": time.time()}
        self.save()

    def done(self, stage):
        return self.d["stages"].get(stage, {}).get("status") == "done"

    def fact(self, key, value=None):
        if value is None:
            return self.d["facts"].get(key)
        self.d["facts"][key] = value
        self.save()

    def decide(self, text):
        self.d["decisions"].append({"ts": time.time(), "text": text})
        self.save()
        print("  DECISION:", text)

class BaseAgent:
    name = "agent"
    def __init__(self, bus: MessageBus, board: Blackboard):
        self.bus, self.board = bus, board
        bus.register(self)

    def _reply(self, msg, status, payload, error, seconds):
        return Message(self.name, msg.sender, msg.action + ".result", payload, status, RUN_ID,
                       reply_to=msg.msg_id, error=error, seconds=seconds)

    def receive(self, msg: Message) -> Message:
        t0 = time.time()
        fn = getattr(self, "act_" + msg.action, None)
        if fn is None:
            return self._reply(msg, "error", {}, f"{self.name} has no action {msg.action!r}", 0.0)
        try:
            out = fn(**msg.payload) or {}
            if not isinstance(out, dict):
                out = {"result": out}
            return self._reply(msg, "done", out, None, time.time() - t0)
        except Exception as e:                                      # noqa: BLE001
            return self._reply(msg, "error", {}, f"{type(e).__name__}: {e}\n" + traceback.format_exc(limit=8), time.time() - t0)

    def ask(self, recipient, action, **payload):
        """Send a request to another agent through the bus."""
        return self.bus.send(Message(self.name, recipient, action, payload))

def show_fig(fig, path):
    """Save a matplotlib figure and display it (report.py switches matplotlib to the Agg backend)."""
    import matplotlib.pyplot as plt
    from IPython.display import Image, display
    fig.savefig(path, dpi=130, bbox_inches="tight"); plt.close(fig)
    display(Image(str(path)))

print("framework ready")

## 4. Monitor, acquisition, preprocessing, detection, feature agents

In [ ]:
def gpu_snapshot():
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total,utilization.gpu", "--format=csv,noheader,nounits"],
                           capture_output=True, text=True, timeout=10).stdout.strip().splitlines()[0]
        used, tot, util = [float(x) for x in r.split(",")]
        return {"gpu_mem_mb": used, "gpu_mem_total_mb": tot, "gpu_util_pct": util}
    except Exception:
        return {"gpu_mem_mb": None, "gpu_mem_total_mb": None, "gpu_util_pct": None}

def ram_gb():
    import psutil
    return round(psutil.virtual_memory().used / 1e9, 2)

class MonitorAgent(BaseAgent):
    """Records timing and GPU/RAM use around every stage."""
    name = "monitor"
    def __init__(self, bus, board):
        super().__init__(bus, board)
        self.path = RUN_DIR / "metrics.jsonl"

    def act_snapshot(self, stage, phase):
        rec = {"ts": time.time(), "stage": stage, "phase": phase, "ram_gb": ram_gb(), **gpu_snapshot()}
        with open(self.path, "a") as f:
            f.write(json.dumps(rec) + "\n")
        return rec

    def act_summary(self):
        if not self.path.exists():
            return {"rows": []}
        recs = [json.loads(l) for l in open(self.path)]
        rows = {}
        for r in recs:
            d = rows.setdefault(r["stage"], {"stage": r["stage"]})
            if r["phase"] == "start":
                d["t0"] = r["ts"]
            else:
                d["seconds"] = round(r["ts"] - d.get("t0", r["ts"]), 1)
                d["gpu_mem_mb_end"], d["ram_gb_end"], d["gpu_util_pct_end"] = r["gpu_mem_mb"], r["ram_gb"], r["gpu_util_pct"]
        return {"rows": [{k: v for k, v in d.items() if k != "t0"} for d in rows.values()]}


class AcquisitionAgent(BaseAgent):
    """Knows what data exists; can fetch more from MAST when asked."""
    name = "acquisition"

    def act_inventory(self):
        n = lambda d: len(list(d.glob("*.npz"))) if d.exists() else 0
        fm = CAT / "feature_matrix.csv"
        inv = {"lc_detrended": n(DET), "lc_raw": n(RAW), "lc_injected": n(INJ),
               "feature_rows": (sum(1 for _ in open(fm)) - 1) if fm.exists() else 0,
               "has_labels": (CAT / "unified_labels.csv").exists(),
               "has_stellar": (CAT / "tic_stellar_params.csv").exists()}
        self.board.fact("inventory", inv)
        return inv

    def act_download(self, tic_ids, max_sectors=2, concurrency=8):
        """Download SPOC 2-minute light curves for new targets (needs internet)."""
        res = acquisition.download_light_curves([str(t) for t in tic_ids], output_dir=RAW,
                                                concurrency=concurrency, max_sectors=max_sectors)
        return {"result": _short(res, 400), "lc_raw": len(list(RAW.glob("*.npz")))}


class PreprocessingAgent(BaseAgent):
    name = "preprocessing"

    def act_preprocess(self, force=False):
        n_raw = len(list(RAW.glob("*.npz"))) if RAW.exists() else 0
        n_inj = len(list(INJ.glob("*.npz"))) if INJ.exists() else 0
        n_det = len(list(DET.glob("*.npz"))) if DET.exists() else 0
        if n_det and (n_raw + n_inj == 0 or n_det >= 0.9 * (n_raw + n_inj)) and not force:
            return {"skipped": True, "detrended": n_det, "reason": "detrended light curves already present"}
        if n_raw + n_inj == 0:
            raise RuntimeError("nothing to preprocess: no lc_raw / lc_injected light curves")
        df = preprocessing.preprocess_batch(RAW, DET, n_jobs=os.cpu_count() or 2, include_injected=True)
        return {"skipped": False, "status_counts": df["status"].value_counts().to_dict(), "detrended": len(list(DET.glob("*.npz")))}


def _detect_one(path):
    import numpy as np, detection
    d = np.load(path, allow_pickle=True)
    t, f = detection.select_search_window(d["time"].astype(float), d["flat_flux"].astype(float))
    r = detection.detect_candidate(t, f, Path(path).stem)
    return r

class DetectionAgent(BaseAgent):
    """Transit Least Squares scan of detrended curves (CPU). Used for new, unlabelled targets."""
    name = "detection"

    def act_scan(self, limit=200, only_missing_features=True):
        from joblib import Parallel, delayed
        files = sorted(DET.glob("*.npz"))
        if only_missing_features and (CAT / "feature_matrix.csv").exists():
            done = set(pd.read_csv(CAT / "feature_matrix.csv", usecols=["tic_id"], dtype=str)["tic_id"])
            files = [f for f in files if f.stem not in done]
        files = files[:limit]
        if not files:
            return {"scanned": 0, "candidates": 0, "note": "every detrended curve already has features"}
        rows = Parallel(n_jobs=os.cpu_count() or 2)(delayed(_detect_one)(str(f)) for f in files)
        df = pd.DataFrame(rows)
        df.to_csv(OUT / "detections.csv", index=False)
        return {"scanned": len(df), "candidates": int(df["is_candidate"].sum()), "csv": str(OUT / "detections.csv")}


class FeatureAgent(BaseAgent):
    """65+ features per light curve. Heavy numerics run on the GPU (CuPy / PyTorch) with CPU fallback."""
    name = "features"

    def act_extract(self, force=False):
        n_det = len(list(DET.glob("*.npz")))
        fm = CAT / "feature_matrix.csv"
        have = 0
        if fm.exists() and not force:
            d = pd.read_csv(fm, usecols=["tic_id", "status"], dtype=str, on_bad_lines="skip")
            have = int((d["status"] == "done").sum())
            if have >= n_det:
                return {"skipped": True, "rows_done": have, "detrended": n_det}
        if n_det == 0:
            raise RuntimeError("no detrended light curves to extract features from")
        features.extract_features_batch(detrended_dir=DET, label_csv=CAT / "unified_labels.csv" if (CAT / "unified_labels.csv").exists() else None,
                                        n_jobs=-1, output_path=fm, resume=not force)
        d = pd.read_csv(fm, usecols=["status"], dtype=str, on_bad_lines="skip")
        return {"skipped": False, "rows_done": int((d["status"] == "done").sum()), "detrended": n_det,
                "gpu": features.gpu_features.get_device_info() if hasattr(features, "gpu_features") else None}

print("monitor, acquisition, preprocessing, detection, features agents defined")

## 5. Classification agent: trainable, GPU, with a model registry

Trains the leakage-safe stacked classifier from `classifier.py` (group-aware splits, synthetic injections in training only, calibrated probabilities), but first **searches XGBoost hyper-parameters with Optuna on the GPU** (grouped CV, objective = PR-AUC on the transit class). Every trained model is versioned in `models/registry.json`; a new version is **promoted to champion only if it is not worse** on the held-out set. Human labels from the active-learning agent (`data/catalogs/human_labels.csv`) override catalogue labels.

In [ ]:
HUMAN_LABELS = CAT / "human_labels.csv"

def norm_id(t):
    return fitting.norm_tic(t)

def load_human_labels():
    if not HUMAN_LABELS.exists():
        return {}
    d = pd.read_csv(HUMAN_LABELS, dtype={"tic_id": str})
    return {norm_id(t): int(l) for t, l in zip(d["tic_id"], d["label"])}

def apply_human_labels(ids, y):
    """Replace catalogue labels by expert labels where available. Returns (y, n_overridden)."""
    hl = load_human_labels()
    y = np.array(y, copy=True)
    n = 0
    for i, t in enumerate(ids):
        k = norm_id(t)
        if k in hl and y[i] != hl[k]:
            y[i], n = hl[k], n + 1
    return y, n

def labelled_rows():
    """Rows of the feature matrix exactly as classifier.load_dataset keeps them (same order)."""
    df = pd.read_csv(CAT / "feature_matrix.csv", skipinitialspace=True)
    df.columns = df.columns.str.strip()
    df = df[(df["status"].astype(str).str.strip() == "done") & (df["label"] >= 0)].copy()
    df["tic_id"] = df["tic_id"].astype(str).str.strip()
    return df.reset_index(drop=True)

def registry():
    p = MODELS / "registry.json"
    return json.loads(p.read_text()) if p.exists() else []

class ClassificationAgent(BaseAgent):
    name = "classification"

    # ---- hyper-parameter search (GPU XGBoost) ---------------------------------
    def _tune(self, X, y, groups, trials, seed):
        import optuna, xgboost as xgb
        from sklearn.model_selection import StratifiedGroupKFold
        from sklearn.metrics import average_precision_score
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        yb = (y == 0).astype(int)                                   # positive class = Transit
        dev = classifier._xgb_device()
        cv = list(StratifiedGroupKFold(3, shuffle=True, random_state=seed).split(X, yb, groups))
        pos_w = float((yb == 0).sum() / max((yb == 1).sum(), 1))

        def objective(trial):
            p = dict(n_estimators=trial.suggest_int("n_estimators", 150, 700, step=50),
                     max_depth=trial.suggest_int("max_depth", 3, 9),
                     learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                     subsample=trial.suggest_float("subsample", 0.6, 1.0),
                     colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
                     min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
                     reg_lambda=trial.suggest_float("reg_lambda", 1e-2, 10.0, log=True))
            scores = []
            for a, b in cv:
                m = xgb.XGBClassifier(**p, tree_method="hist", device=dev, eval_metric="logloss", random_state=seed,
                                      scale_pos_weight=pos_w, n_jobs=os.cpu_count() or 2)
                m.fit(X.iloc[a], yb[a])
                scores.append(average_precision_score(yb[b], m.predict_proba(X.iloc[b])[:, 1]))
            return float(np.mean(scores))

        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
        study.optimize(objective, n_trials=trials)
        return study.best_params, float(study.best_value), dev

    def act_train(self, trials=40, use_smote=False, seed=42, note=""):
        from sklearn.metrics import roc_auc_score
        X, y, groups, is_syn = classifier.load_dataset(CAT / "feature_matrix.csv")
        df = labelled_rows()
        assert len(df) == len(y), "row alignment between labelled_rows() and classifier.load_dataset() broke"
        y, n_over = apply_human_labels(df["tic_id"].to_numpy(), y)
        tr, te = classifier.group_split(y, groups, is_syn)

        params, cv_pr, dev = ({}, None, "cpu")
        if trials > 0:
            try:
                params, cv_pr, dev = self._tune(X.iloc[tr], y[tr], groups[tr], trials, seed)
            except Exception as e:                                  # noqa: BLE001
                print("   (tuning skipped:", type(e).__name__, e, ")")

        orig = classifier.make_base_learners
        def patched(use_smote=False, n_jobs=2, _p=dict(params)):
            L = orig(use_smote, n_jobs)
            if "xgb" in L and _p:
                est = L["xgb"]
                est.set_params(**({f"clf__{k}": v for k, v in _p.items()} if hasattr(est, "named_steps") else _p))
            return L
        classifier.make_base_learners = patched
        try:
            model = classifier.StackedClassifier(use_smote=use_smote, n_jobs=os.cpu_count() or 2).fit(X.iloc[tr], y[tr], groups[tr])
        finally:
            classifier.make_base_learners = orig

        ev = classifier.evaluate(model, X.iloc[te], y[te])
        top = classifier.explain(model, X.iloc[te], y[te])
        classifier._xgb_to_cpu(model)                                # keep the pickle portable

        # ---- registry: version, compare with champion, promote if not worse ----
        reg = registry()
        ver = len(reg) + 1
        vdir = MODELS / f"v{ver}"
        vdir.mkdir(parents=True, exist_ok=True)
        import joblib
        joblib.dump(model, vdir / "classifier.joblib")
        champion = next((r for r in reg if r.get("champion")), None)
        promote = champion is None or ev["pr_auc"] >= champion["pr_auc"] - 1e-9
        entry = {"version": ver, "ts": time.time(), "pr_auc": ev["pr_auc"], "roc_auc": ev["roc_auc"], "brier": ev["brier"],
                 "accuracy": ev["report"]["accuracy"], "macro_f1": ev["report"]["macro avg"]["f1-score"],
                 "xgb_params": params, "xgb_device": dev, "cv_pr_auc": cv_pr, "trials": trials, "smote": use_smote,
                 "human_overrides": n_over, "n_train": int(len(tr)), "n_test": int(len(te)), "note": note, "champion": bool(promote)}
        if promote:
            for r in reg:
                r["champion"] = False
            shutil.copy(vdir / "classifier.joblib", MODELS / "classifier.joblib")
            (MODELS / "metrics.json").write_text(json.dumps({"test": ev, "top_features": top, "version": ver}, indent=2, default=str))
        reg.append(entry)
        (MODELS / "registry.json").write_text(json.dumps(reg, indent=1, default=str))
        (vdir / "metrics.json").write_text(json.dumps({"test": ev, "top_features": top}, indent=2, default=str))

        if promote or not (OUT / "tab_predictions.csv").exists():      # downstream agents always see the champion's scores
            self._predict_all(model, df, tr, te).to_csv(OUT / "tab_predictions.csv", index=False)
        return {"version": ver, "promoted": bool(promote), "pr_auc": ev["pr_auc"], "roc_auc": ev["roc_auc"], "brier": ev["brier"],
                "accuracy": ev["report"]["accuracy"], "macro_f1": ev["report"]["macro avg"]["f1-score"], "xgb_device": dev,
                "tuned_params": params, "human_overrides": n_over, "top_features": [t.get("feature") for t in top[:8]]}

    def _predict_all(self, model, df_lab, tr, te):
        """p(Transit) for every successfully extracted row, tagged train / test / unlabelled."""
        fm = pd.read_csv(CAT / "feature_matrix.csv", skipinitialspace=True)
        fm.columns = fm.columns.str.strip()
        fm = fm[fm["status"].astype(str).str.strip() == "done"].copy()
        fm["tic_id"] = fm["tic_id"].astype(str).str.strip()
        Xall = fm.reindex(columns=model.feature_names_).astype(float).replace([np.inf, -np.inf], np.nan)
        p = model.predict_proba(Xall)[:, list(model.classes_).index(0)]
        te_ids, tr_ids = set(df_lab["tic_id"].iloc[te]), set(df_lab["tic_id"].iloc[tr])
        split = np.where(fm["tic_id"].isin(te_ids), "test", np.where(fm["tic_id"].isin(tr_ids), "train", "unlabelled"))
        return pd.DataFrame({"tic_id": fm["tic_id"].to_numpy(), "split": split,
                             "y": np.where(fm["label"] >= 0, fm["label"], np.nan), "p_tab": p})

    def act_registry(self):
        return {"versions": registry()}

print("classification agent defined")

## 6. Deep-vetting agent: 1D CNN trained on the GPU

A second, independent opinion. It never sees the engineered features: each candidate is phase-folded at its TLS period into a **global view** (whole orbit, 1001 bins) and a **local view** (2.5 transit durations either side, 201 bins), normalised so the dip is -1, and fed to a two-branch CNN. Training uses mixed precision (AMP) on the GPU, transit-symmetric augmentation (time flip, noise), class-weighted loss, a star-grouped validation split for early stopping, and the **same held-out test stars as the tabular classifier**, so the two agents can be compared and combined fairly.

In [ ]:
G_BINS, L_BINS = 1001, 201

def _bin_mean(x, y, lo, hi, n):
    idx = np.floor((x - lo) / (hi - lo) * n).astype(int)
    ok = (idx >= 0) & (idx < n)
    if ok.sum() < 20:
        return None
    s = np.bincount(idx[ok], weights=y[ok], minlength=n)
    c = np.bincount(idx[ok], minlength=n)
    m = c > 0
    if m.sum() < 5:
        return None
    out = np.full(n, np.nan)
    out[m] = s[m] / c[m]
    xs = np.arange(n)
    out[~m] = np.interp(xs[~m], xs[m], out[m])
    return out

def _fold_views(t, f, P, t0, dur_hr):
    ph = ((t - t0 + 0.5 * P) % P) - 0.5 * P
    g = _bin_mean(ph, f, -0.5 * P, 0.5 * P, G_BINS)
    w = min(max(2.5 * dur_hr / 24.0, 0.02), 0.5 * P)
    l = _bin_mean(ph, f, -w, w, L_BINS)
    if g is None or l is None:
        return None
    def norm(v):
        v = v - np.median(v)
        return (v / max(abs(v.min()), 1e-6)).astype(np.float32)
    return norm(g), norm(l)

def _views_one(args):
    tic, P, t0, dur_hr = args
    import numpy as np, fitting
    try:
        seg = fitting.load_segment(tic)
        t, f = seg["time"], seg["flat"]
        ok = np.isfinite(t) & np.isfinite(f)
        return _fold_views(t[ok], f[ok], float(P), float(t0), float(dur_hr))
    except Exception:
        return None

import torch, torch.nn as nn

class TwoBranchCNN(nn.Module):
    def __init__(self, g=G_BINS, l=L_BINS):
        super().__init__()
        def branch():
            layers, c = [], 1
            for o in (16, 32, 64, 128):
                layers += [nn.Conv1d(c, o, 5, padding=2), nn.BatchNorm1d(o), nn.ReLU(), nn.MaxPool1d(2)]
                c = o
            return nn.Sequential(*layers, nn.AdaptiveAvgPool1d(4), nn.Flatten())
        self.g, self.l = branch(), branch()
        self.head = nn.Sequential(nn.Linear(2 * 128 * 4, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 1))

    def forward(self, xg, xl):
        return self.head(torch.cat([self.g(xg), self.l(xl)], dim=1)).squeeze(1)


class DeepVettingAgent(BaseAgent):
    name = "deep_vetting"

    def _views(self, df, rows):
        """(G, L, valid) arrays for the requested row indices; cached on disk."""
        from joblib import Parallel, delayed
        key = hashlib.md5(",".join(df["tic_id"].iloc[rows]).encode()).hexdigest()[:10]
        cache = WS / "cache" / f"views_{len(rows)}_{key}.npz"
        Gv = np.zeros((len(df), G_BINS), np.float32); Lv = np.zeros((len(df), L_BINS), np.float32)
        valid = np.zeros(len(df), bool)
        if cache.exists():
            z = np.load(cache)
            return z["G"], z["L"], z["valid"]
        args = [(df["tic_id"].iloc[i], df["period"].iloc[i], df["t0"].iloc[i], df["duration_hr"].iloc[i]) for i in rows]
        ok_rows = [k for k, a in zip(rows, args) if np.isfinite(a[1]) and np.isfinite(a[2]) and np.isfinite(a[3]) and a[1] > 0]
        args_ok = [(df["tic_id"].iloc[i], df["period"].iloc[i], df["t0"].iloc[i], df["duration_hr"].iloc[i]) for i in ok_rows]
        out = Parallel(n_jobs=os.cpu_count() or 2)(delayed(_views_one)(a) for a in args_ok)
        for i, v in zip(ok_rows, out):
            if v is not None:
                Gv[i], Lv[i], valid[i] = v[0], v[1], True
        np.savez_compressed(cache, G=Gv, L=Lv, valid=valid)
        return Gv, Lv, valid

    def act_train(self, epochs=40, max_rows=0, batch=128, seed=42, lr=2e-3):
        from sklearn.metrics import roc_auc_score, average_precision_score
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        if dev == "cpu":
            epochs = min(epochs, 8)
        torch.manual_seed(seed); np.random.seed(seed)
        df = labelled_rows()
        X, y, groups, is_syn = classifier.load_dataset(CAT / "feature_matrix.csv")
        y, _ = apply_human_labels(df["tic_id"].to_numpy(), y)
        tr, te = classifier.group_split(y, groups, is_syn)
        rng = np.random.default_rng(seed)
        if max_rows and len(df) > max_rows:
            tr = rng.choice(tr, min(int(max_rows * 0.75), len(tr)), replace=False)
            te = rng.choice(te, min(max_rows - int(max_rows * 0.75), len(te)), replace=False)
            rows = np.concatenate([tr, te])
        else:
            rows = np.arange(len(df))
        t0 = time.time()
        Gv, Lv, valid = self._views(df, rows)
        t_views = time.time() - t0
        yb = (y == 0).astype(np.float32)
        tr = tr[valid[tr]]; te = te[valid[te]]
        if len(tr) < 50 or len(np.unique(yb[tr])) < 2:
            raise RuntimeError(f"too few valid training views ({len(tr)}); is lc_detrended attached and does feature_matrix have period/t0?")

        # star-grouped validation split for early stopping
        ug = np.unique(groups[tr]); rng.shuffle(ug)
        vg = set(ug[: max(2, int(0.1 * len(ug)))])
        is_val = np.array([g in vg for g in groups[tr]])
        fit_idx, val_idx = tr[~is_val], tr[is_val]
        if len(np.unique(yb[val_idx])) < 2:                       # degenerate val: fall back to a random tail
            cut = int(0.9 * len(tr)); fit_idx, val_idx = tr[:cut], tr[cut:]

        toT = lambda a: torch.tensor(a).unsqueeze(1).to(dev)
        Gt, Lt, Yt = toT(Gv), toT(Lv), torch.tensor(yb).to(dev)
        net = TwoBranchCNN().to(dev)
        opt = torch.optim.AdamW(net.parameters(), lr=lr / 2, weight_decay=1e-4)
        steps_per_epoch = int(math.ceil(len(fit_idx) / batch))
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=max(epochs * steps_per_epoch, 1))
        pos_w = torch.tensor(float((yb[fit_idx] == 0).sum() / max((yb[fit_idx] == 1).sum(), 1)), device=dev)
        lossf = nn.BCEWithLogitsLoss(pos_weight=pos_w)
        scaler = torch.cuda.amp.GradScaler(enabled=(dev == "cuda"))

        @torch.no_grad()
        def predict(idx):
            net.eval(); out = []
            for i in range(0, len(idx), 512):
                b = torch.tensor(idx[i:i + 512], device=dev)
                with torch.autocast(device_type=dev, dtype=torch.float16, enabled=(dev == "cuda")):
                    out.append(torch.sigmoid(net(Gt[b], Lt[b])).float().cpu().numpy())
            return np.concatenate(out) if out else np.array([])

        best, best_state, bad, hist = -1.0, None, 0, []
        fit_t = torch.tensor(fit_idx, device=dev)
        for ep in range(epochs):
            net.train(); perm = fit_t[torch.randperm(len(fit_t), device=dev)]; tot = 0.0
            for i in range(0, len(perm), batch):
                b = perm[i:i + batch]
                xg, xl, yy = Gt[b], Lt[b], Yt[b]
                flip = torch.rand(len(b), 1, 1, device=dev) < 0.5     # a transit is time-symmetric
                xg = torch.where(flip, xg.flip(-1), xg); xl = torch.where(flip, xl.flip(-1), xl)
                xg = xg + 0.02 * torch.randn_like(xg); xl = xl + 0.02 * torch.randn_like(xl)
                opt.zero_grad(set_to_none=True)
                with torch.autocast(device_type=dev, dtype=torch.float16, enabled=(dev == "cuda")):
                    loss = lossf(net(xg, xl), yy)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
                tot += float(loss) * len(b)
            pv = predict(val_idx)
            vpr = average_precision_score(yb[val_idx], pv) if len(np.unique(yb[val_idx])) > 1 else 0.0
            hist.append({"epoch": ep + 1, "loss": tot / len(fit_idx), "val_pr_auc": float(vpr)})
            if vpr > best:
                best, bad = vpr, 0
                best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
            else:
                bad += 1
                if bad >= 8:
                    break
        net.load_state_dict(best_state)

        # ---- evaluate on the held-out stars, predict every valid row ----
        pt = predict(te)
        met = {"roc_auc": float(roc_auc_score(yb[te], pt)), "pr_auc": float(average_precision_score(yb[te], pt)),
               "accuracy": float(((pt > 0.5) == (yb[te] > 0.5)).mean()), "n_test": int(len(te))} if len(np.unique(yb[te])) > 1 else {}
        allv = np.where(valid)[0]
        p_all = np.full(len(df), np.nan); p_all[allv] = predict(allv)
        split = np.full(len(df), "unused", dtype=object); split[tr] = "train"; split[te] = "test"
        pd.DataFrame({"tic_id": df["tic_id"], "split_cnn": split, "p_cnn": p_all}).to_csv(OUT / "cnn_predictions.csv", index=False)
        n = len(list(MODELS.glob("cnn_v*.pt"))) + 1
        torch.save(best_state, MODELS / f"cnn_v{n}.pt")
        del Gt, Lt, Yt, net
        torch.cuda.empty_cache()
        return {"device": dev, "epochs_run": len(hist), "views_seconds": round(t_views, 1), "val_pr_auc": float(best),
                "history_last": hist[-1], **{f"test_{k}": v for k, v in met.items()}, "weights": f"cnn_v{n}.pt"}

print("deep-vetting agent defined")

## 7. Consensus agent

Combines the tabular and CNN opinions. It uses the plain mean of the two probabilities (no weights tuned on the test set, so nothing leaks) and reports each agent's and the ensemble's held-out metrics, plus a **disagreement** score `|p_tab - p_cnn|` that flags candidates the agents cannot agree on. If the CNN did not run, it degrades gracefully to the tabular score alone.

In [ ]:
class ConsensusAgent(BaseAgent):
    name = "consensus"

    def act_build(self, min_prob=0.5):
        from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
        tab = pd.read_csv(OUT / "tab_predictions.csv", dtype={"tic_id": str})
        cnn_path = OUT / "cnn_predictions.csv"
        if cnn_path.exists():
            cnn = pd.read_csv(cnn_path, dtype={"tic_id": str})[["tic_id", "p_cnn"]]
            df = tab.merge(cnn, on="tic_id", how="left")
        else:
            df = tab.assign(p_cnn=np.nan)
        both = df["p_tab"].notna() & df["p_cnn"].notna()
        df["p_consensus"] = np.where(both, 0.5 * (df["p_tab"] + df["p_cnn"]), df["p_tab"])
        df["disagreement"] = np.where(both, (df["p_tab"] - df["p_cnn"]).abs(), np.nan)
        df.to_csv(OUT / "predictions.csv", index=False)

        te = df[(df["split"] == "test") & df["y"].notna()]
        def m(col, sub):
            s = sub[sub[col].notna()]
            yb = (s["y"] == 0).astype(int)
            if len(s) < 10 or yb.nunique() < 2:
                return None
            pred = (s[col] >= 0.5).astype(int)
            return {"n": int(len(s)), "roc_auc": float(roc_auc_score(yb, s[col])), "pr_auc": float(average_precision_score(yb, s[col])),
                    "accuracy": float((pred == yb).mean()), "macro_f1": float(f1_score(yb, pred, average="macro"))}
        same = te[te["p_cnn"].notna()]                               # compare all three on identical rows
        res = {"tabular": m("p_tab", same if len(same) else te), "cnn": m("p_cnn", same), "consensus": m("p_consensus", same if len(same) else te)}
        (OUT / "consensus_metrics.json").write_text(json.dumps(res, indent=1))
        real = df[~df["tic_id"].str.startswith("SYN")]
        n_cand = int((real["p_consensus"] >= min_prob).sum())
        return {"metrics": res, "n_candidates": n_cand, "cnn_used": bool(both.any()),
                "mean_disagreement": float(df["disagreement"].mean()) if both.any() else None}

print("consensus agent defined")

## 8. Active-learning agent

Human labels are the scarcest resource, so this agent decides **which candidates an expert should label next**. It has two jobs:

1. `queue`: writes `outputs/review_queue.csv` of the most informative candidates (uncertain or disputed by the two agents). A human fills the `human_label` column (`transit` or `other`), and `ingest` merges those labels into `data/catalogs/human_labels.csv`, after which the orchestrator retrains the classifier.
2. `simulate`: an experiment that **measures the value of the query policy**. Starting from a small labelled set, it repeatedly asks an oracle (the known labels) for a batch, comparing three policies: `random`, `uncertainty` (closest to 0.5) and `committee` (an XGBoost model and a gradient-boosting model disagree). The result is a label-efficiency curve on the held-out stars.

In [ ]:
class ActiveLearningAgent(BaseAgent):
    name = "active_learning"

    def act_queue(self, k=25):
        p = pd.read_csv(OUT / "predictions.csv", dtype={"tic_id": str})
        real = p[~p["tic_id"].str.startswith("SYN")].copy()
        pool = real[real["split"] == "unlabelled"]
        note = "unlabelled targets"
        if len(pool) < k:                                             # nothing unlabelled: demonstrate on held-out rows
            pool = real[real["split"] == "test"]
            note = "held-out test rows (no unlabelled targets in this feature matrix)"
        pool = pool.assign(margin=(pool["p_consensus"] - 0.5).abs())
        score = pool["margin"] - 0.5 * pool["disagreement"].fillna(0)   # uncertain, and disputed = more informative
        q = pool.loc[score.sort_values().index[:k], ["tic_id", "p_tab", "p_cnn", "p_consensus", "disagreement"]].copy()
        q["human_label"] = ""
        q.to_csv(OUT / "review_queue.csv", index=False)
        return {"queued": int(len(q)), "source": note, "csv": str(OUT / "review_queue.csv")}

    def act_ingest(self, path):
        d = pd.read_csv(path, dtype={"tic_id": str}).dropna(subset=["human_label"])
        mp = {"transit": 0, "planet": 0, "0": 0, "other": 3, "fp": 3, "false positive": 3, "3": 3}
        d["label"] = d["human_label"].astype(str).str.strip().str.lower().map(mp)
        d = d.dropna(subset=["label"])
        d["label"] = d["label"].astype(int)
        if HUMAN_LABELS.exists():
            d = pd.concat([pd.read_csv(HUMAN_LABELS, dtype={"tic_id": str}), d[["tic_id", "label"]]]).drop_duplicates("tic_id", keep="last")
        d[["tic_id", "label"]].to_csv(HUMAN_LABELS, index=False)
        return {"labels_total": int(len(d)), "path": str(HUMAN_LABELS)}

    def act_simulate(self, rounds=8, batch=25, seeds=3, init=60):
        import xgboost as xgb
        from sklearn.ensemble import HistGradientBoostingClassifier
        from sklearn.metrics import average_precision_score, roc_auc_score
        X, y, groups, is_syn = classifier.load_dataset(CAT / "feature_matrix.csv")
        tr, te = classifier.group_split(y, groups, is_syn)
        tr = tr[~is_syn[tr]]                                          # real stars only
        yb = (y == 0).astype(int)
        dev = classifier._xgb_device()
        Xte, yte = X.iloc[te], yb[te]
        def fit_xgb(idx):
            m = xgb.XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.1, tree_method="hist", device=dev,
                                  eval_metric="logloss", n_jobs=os.cpu_count() or 2, random_state=0)
            return m.fit(X.iloc[idx], yb[idx])
        def fit_hgb(idx):
            return HistGradientBoostingClassifier(max_iter=100, random_state=0).fit(X.iloc[idx], yb[idx])
        curves = {s: [] for s in ("random", "uncertainty", "committee")}
        init = min(init, max(len(tr) // 4, 10))
        for seed in range(seeds):
            rng = np.random.default_rng(seed)
            first = np.concatenate([rng.choice(tr[yb[tr] == c], size=max(init // 2, 2), replace=False) for c in (0, 1)])
            for strat in curves:
                fs = set(first.tolist())
                lab, pool = list(first), np.array([i for i in tr if i not in fs])
                pts = []
                for r in range(rounds + 1):
                    m = fit_xgb(np.array(lab))
                    p_te = m.predict_proba(Xte)[:, 1]
                    pts.append((len(lab), float(average_precision_score(yte, p_te)), float(roc_auc_score(yte, p_te))))
                    if r == rounds or len(pool) < batch:
                        break
                    if strat == "random":
                        pick = rng.choice(len(pool), batch, replace=False)
                    else:
                        px = m.predict_proba(X.iloc[pool])[:, 1]
                        if strat == "uncertainty":
                            score = np.abs(px - 0.5)
                        else:
                            ph = fit_hgb(np.array(lab)).predict_proba(X.iloc[pool])[:, 1]
                            score = -np.abs(px - ph)                  # most disagreement first
                        pick = np.argsort(score)[:batch]
                    lab += list(pool[pick]); pool = np.delete(pool, pick)
                curves[strat].append(pts)
        agg = {}
        for s, runs in curves.items():
            L = min(len(r) for r in runs)
            arr = np.array([[pt[1] for pt in r[:L]] for r in runs])
            agg[s] = {"n_labels": [pt[0] for pt in runs[0][:L]], "pr_auc_mean": arr.mean(0).tolist(), "pr_auc_std": arr.std(0).tolist()}
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(7, 4))
        for s, col in (("random", "#8d99ae"), ("uncertainty", "#2a9d8f"), ("committee", "#e76f51")):
            a = agg[s]; mu, sd = np.array(a["pr_auc_mean"]), np.array(a["pr_auc_std"])
            ax.plot(a["n_labels"], mu, "o-", color=col, label=s); ax.fill_between(a["n_labels"], mu - sd, mu + sd, color=col, alpha=0.15)
        ax.set_xlabel("Expert labels used"); ax.set_ylabel("Held-out PR-AUC (Transit)"); ax.legend(); ax.set_title("Active-learning label efficiency")
        show_fig(fig, OUT / "al_curve.png")
        rnd_final = agg["random"]["pr_auc_mean"][-1]
        saving = {}
        for s in ("uncertainty", "committee"):
            hit = [n for n, v in zip(agg[s]["n_labels"], agg[s]["pr_auc_mean"]) if v >= rnd_final]
            saving[s] = (agg["random"]["n_labels"][-1] - hit[0]) if hit else 0
        (OUT / "al_results.json").write_text(json.dumps({"curves": agg, "labels_saved_vs_random": saving}, indent=1))
        return {"final_pr_auc": {s: agg[s]["pr_auc_mean"][-1] for s in agg}, "labels_saved_vs_random": saving, "plot": str(OUT / "al_curve.png")}

print("active-learning agent defined")

## 9. Fitting, reporting and explainer agents

* **fitting**: picks the top candidates by consensus probability (held-out and unlabelled stars first, so training-set scores are not passed off as discoveries), fits a batman transit model with emcee, and **self-heals**: any fit that did not converge is automatically re-run with a longer chain.
* **reporting**: 7 diagnostic plots, vetting checks (odd/even, secondary eclipse, centroid, size, SNR), a ranked catalogue and a 3-page PDF per candidate.
* **explainer**: turns each candidate's numbers into a short plain-English assessment with an open LLM on the GPU. The LLM is only shown the fitted facts and its output is checked against them; if it fails the check, or the model cannot load, a deterministic template is used instead.

In [ ]:
class FittingAgent(BaseAgent):
    name = "fitting"

    def _candidates(self, top, min_prob, prefer_heldout):
        pred = pd.read_csv(OUT / "predictions.csv", dtype={"tic_id": str})
        real = pred[~pred["tic_id"].str.startswith("SYN")]
        cand = real[real["p_consensus"] >= min_prob].sort_values("p_consensus", ascending=False)
        if prefer_heldout:
            cand = pd.concat([cand[cand["split"].isin(["test", "unlabelled"])], cand[cand["split"] == "train"]])
        cand = cand.head(top)
        fm = pd.read_csv(CAT / "feature_matrix.csv", skipinitialspace=True)
        fm.columns = fm.columns.str.strip()
        fm["tic_id"] = fm["tic_id"].astype(str).str.strip()
        c = cand[["tic_id", "p_consensus", "split"]].merge(fm, on="tic_id", how="left").rename(columns={"p_consensus": "p_transit"})
        c = c[c["period"].notna() & c["t0"].notna() & c["duration_hr"].notna()].copy()
        c["tic_id"] = c["tic_id"].map(fitting.norm_tic)
        return c

    def _rebuild(self, cands):
        rows = []
        for t in cands["tic_id"]:
            p = OUT / "fits" / f"{t}.json"
            if p.exists():
                rows.append(fitting.flatten(json.loads(p.read_text())))
        tab = pd.DataFrame(rows).merge(cands[["tic_id", "p_transit"]], on="tic_id", how="left")
        tab.to_csv(OUT / "fit_results.csv", index=False)
        return tab

    def act_fit(self, top=30, min_prob=0.5, steps=1000, max_steps=4000, n_jobs=4, prefer_heldout=True, force=False, heal=True):
        cands = self._candidates(top, min_prob, prefer_heldout)
        if cands.empty:
            raise RuntimeError(f"no candidates with consensus probability >= {min_prob}")
        tab = fitting.fit_batch(cands, OUT, n_jobs=n_jobs, force=force, n_steps=steps, max_steps=max_steps)
        bad = tab[(tab["status"] == "done") & (tab["method"] != "mcmc")]["tic_id"].tolist()
        healed = 0
        if heal and bad:
            print(f"   self-heal: {len(bad)} fit(s) did not converge -> refitting with up to {max_steps * 2} steps")
            fitting.fit_batch(cands[cands["tic_id"].isin(bad)], OUT, n_jobs=n_jobs, force=True, n_steps=steps, max_steps=max_steps * 2)
            tab = self._rebuild(cands)
            healed = len(bad) - int(((tab["tic_id"].isin(bad)) & (tab["method"] != "mcmc")).sum())
        else:
            tab = self._rebuild(cands)
        return {"candidates": int(len(cands)), "fitted": int((tab["status"] == "done").sum()),
                "mcmc": int((tab["method"] == "mcmc").sum()), "non_converged_first_pass": len(bad), "healed": healed,
                "min_prob": min_prob}


class ReportingAgent(BaseAgent):
    name = "reporting"

    def act_report(self, n_jobs=4, pdf=True):
        cat = report.run(OUT, tics=None, pdf=pdf, n_jobs=n_jobs)
        pred = pd.read_csv(OUT / "predictions.csv", dtype={"tic_id": str})[["tic_id", "split", "p_tab", "p_cnn", "disagreement"]]
        cat = cat.merge(pred, on="tic_id", how="left")
        cat.to_csv(OUT / "candidate_catalog.csv", index=False)
        return {"rows": int(len(cat)), "verdicts": cat["verdict"].value_counts().to_dict(),
                "pdfs": len(list((OUT / "reports").glob("*.pdf"))), "catalog": str(OUT / "candidate_catalog.csv")}


def _num(r, k, fmt="{:.3g}"):
    v = r.get(k)
    return fmt.format(v) if v is not None and v == v else "n/a"

def template_summary(r):
    flags = r.get("flags") if isinstance(r.get("flags"), str) and r.get("flags") else "none"
    txt = (f"{r['tic_id']} is classified {r['verdict']}. The fitted period is {_num(r, 'period_d', '{:.5f}')} d with a depth of "
           f"{_num(r, 'depth_ppm', '{:.0f}')} ppm, implying a radius of {_num(r, 'rp_rearth', '{:.1f}')} Earth radii "
           f"({r.get('size_class', 'n/a')}) at SNR {_num(r, 'snr', '{:.1f}')}. The classifiers give P(transit) = {_num(r, 'p_transit', '{:.2f}')}"
           f" (tabular {_num(r, 'p_tab', '{:.2f}')}, CNN {_num(r, 'p_cnn', '{:.2f}')}). Vetting flags: {flags}.")
    v = str(r.get("verdict"))
    rec = ("Recommendation: prioritise for ground-based follow-up." if v == "PASS" and (r.get("snr") or 0) >= 10 else
           "Recommendation: promising but needs more transits or a cleaner fit before follow-up." if v in ("PASS", "CAUTION") else
           "Recommendation: likely not a planet; deprioritise.")
    return txt + " " + rec

class ExplainerAgent(BaseAgent):
    name = "explainer"
    llm = tok = None
    SYSTEM = ("You are a careful exoplanet-vetting assistant. Write a 4 to 6 sentence plain-English assessment for an astronomer "
              "using ONLY the facts in the JSON. Do not invent numbers, do not claim the object is confirmed, mention the verdict and any "
              "flags, and finish with one line starting 'Recommendation:'.")

    def _load(self, llm_id):
        from transformers import AutoTokenizer, AutoModelForCausalLM
        gpu = torch.cuda.is_available()
        self.tok = AutoTokenizer.from_pretrained(llm_id)
        self.llm = AutoModelForCausalLM.from_pretrained(llm_id, torch_dtype=torch.float16 if gpu else torch.float32,
                                                        device_map="auto" if gpu else None)

    def _gen(self, system, user, max_new=230):
        msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        text = self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp = self.tok(text, return_tensors="pt").to(self.llm.device)
        with torch.no_grad():
            out = self.llm.generate(**inp, max_new_tokens=max_new, do_sample=False)
        return self.tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    def act_release(self):
        self.llm = self.tok = None
        import gc; gc.collect(); torch.cuda.empty_cache()
        return {"released": True}

    def act_explain(self, top=10, use_llm=True, llm_id="Qwen/Qwen2.5-3B-Instruct"):
        cat = pd.read_csv(OUT / "candidate_catalog.csv").head(top)
        mode = "template"
        if use_llm:
            try:
                if self.llm is None:
                    self._load(llm_id)
                mode = "llm"
            except Exception as e:                                          # noqa: BLE001
                print("   LLM unavailable, using templates:", type(e).__name__, str(e)[:120])
        keep = ["tic_id", "verdict", "flags", "period_d", "depth_ppm", "rp_rearth", "size_class", "snr", "teq_k", "in_hz",
                "p_transit", "p_tab", "p_cnn", "disagreement", "odd_even_sigma", "secondary_sigma", "centroid_sigma", "fit_method"]
        out, n_fallback = [], 0
        for _, row in cat.iterrows():
            r = {k: (None if (isinstance(row.get(k), float) and row.get(k) != row.get(k)) else row.get(k)) for k in keep if k in row.index}
            r = {k: (v.item() if hasattr(v, "item") else v) for k, v in r.items()}
            text, src = template_summary(r), "template"
            if mode == "llm":
                try:
                    ans = self._gen(self.SYSTEM, json.dumps(r, default=str))
                    if str(r["verdict"]).lower() in ans.lower() and "recommendation" in ans.lower():
                        text, src = ans, "llm"
                    else:
                        n_fallback += 1
                except Exception:                                           # noqa: BLE001
                    n_fallback += 1
            out.append({"tic_id": r["tic_id"], "verdict": r["verdict"], "source": src, "summary": text})
        (OUT / "candidate_summaries.json").write_text(json.dumps(out, indent=1))
        md_lines = ["# Candidate summaries", ""] + [f"### {o['tic_id']}  ·  {o['verdict']}  ·  _{o['source']}_\n\n{o['summary']}\n" for o in out]
        (OUT / "candidate_summaries.md").write_text("\n".join(md_lines))
        return {"mode": mode, "n": len(out), "llm_rejected_by_check": n_fallback, "file": str(OUT / "candidate_summaries.md")}

    def act_ask(self, question, llm_id="Qwen/Qwen2.5-3B-Instruct", top=15):
        if self.llm is None:
            self._load(llm_id)
        cat = pd.read_csv(OUT / "candidate_catalog.csv").head(top)
        cols = [c for c in ["rank", "tic_id", "verdict", "p_transit", "period_d", "depth_ppm", "rp_rearth", "snr", "in_hz", "flags"] if c in cat]
        sys_p = "You answer questions about exoplanet candidates using ONLY the CSV table given. If the table does not contain the answer, say so."
        return {"answer": self._gen(sys_p, f"TABLE:\n{cat[cols].round(3).to_csv(index=False)}\nQUESTION: {question}", 300)}

print("fitting, reporting, explainer agents defined")

## 10. Orchestrator

The orchestrator does not just run stages in order. After each stage it **observes the result and decides**, and every decision is written to the blackboard:

| Observation | Decision |
|---|---|
| only raw curves present | schedule preprocessing to build `lc_detrended` |
| features already cover the curves | skip feature extraction |
| tabular PR-AUC below 0.60 (full run) | retrain with twice the Optuna budget plus SMOTE, keep the better model |
| no GPU | train the CNN on CPU for at most 8 epochs |
| CNN stage fails | continue in degraded mode with the tabular agent only |
| no candidates above the threshold | lower the threshold to 0.35 once |
| fits that did not converge | refit only those with a longer chain |

Failed calls are retried; optional stages (CNN, explainer, active learning) never stop the run. State is saved after every stage, so an interrupted run resumes where it stopped.

In [ ]:
class Orchestrator(BaseAgent):
    name = "orchestrator"
    OPTIONAL = {"detect", "train_cnn", "explain", "active_learning"}

    def call(self, recipient, action, retries=1, backoff=3.0, **payload):
        r = None
        for attempt in range(retries + 1):
            r = self.ask(recipient, action, **payload)
            if r.status == "done":
                return r.payload
            if attempt < retries:
                self.board.decide(f"{recipient}.{action} failed ({(r.error or '').splitlines()[0][:120]}); retry {attempt + 1}/{retries} in {backoff}s")
                time.sleep(backoff)
        raise RuntimeError(f"{recipient}.{action} failed: {r.error}")

    # ------------------------------------------------------------------ stages
    def s_inventory(self):
        inv = self.call("acquisition", "inventory")
        if inv["lc_detrended"] + inv["lc_raw"] == 0 and inv["feature_rows"] == 0:
            raise RuntimeError("no light curves and no feature matrix: attach data (see the notebook header)")
        if inv["lc_detrended"] == 0 and inv["lc_raw"] > 0:
            self.board.decide("only raw light curves present -> preprocessing agent will build lc_detrended (CPU)")
        if inv["feature_rows"] >= inv["lc_detrended"] > 0:
            self.board.decide("feature matrix already covers the detrended curves -> feature extraction will be skipped")
        return inv

    def s_preprocess(self):
        return self.call("preprocessing", "preprocess")

    def s_detect(self):
        r = self.call("detection", "scan", limit=200)
        if r.get("scanned"):
            self.board.decide(f"{r['candidates']} of {r['scanned']} new curves passed the TLS SDE>=7 cut; feature extraction will process them")
        return r

    def s_features(self):
        return self.call("features", "extract")

    def s_train_tabular(self):
        r = self.call("classification", "train", trials=CFG.optuna_trials, seed=CFG.seed)
        if not CFG.smoke and r["pr_auc"] < 0.60:
            self.board.decide(f"tabular PR-AUC {r['pr_auc']:.3f} < 0.60 -> retraining with {CFG.optuna_trials * 2} trials + SMOTE, keeping the better model")
            r2 = self.call("classification", "train", trials=CFG.optuna_trials * 2, use_smote=True, seed=CFG.seed, note="retrain after low PR-AUC")
            r = r2 if r2["pr_auc"] > r["pr_auc"] else r
        return r

    def s_train_cnn(self):
        if not torch.cuda.is_available():
            self.board.decide("no GPU visible -> CNN will train on CPU with at most 8 epochs")
        try:
            return self.call("deep_vetting", "train", retries=0, epochs=CFG.cnn_epochs, max_rows=CFG.cnn_max_rows, seed=CFG.seed)
        except Exception:
            self.board.decide("CNN stage failed -> continuing in degraded mode: consensus will use the tabular agent only")
            (OUT / "cnn_predictions.csv").unlink(missing_ok=True)
            raise

    def s_consensus(self):
        thr = self.board.fact("min_prob") or CFG.min_prob
        r = self.call("consensus", "build", min_prob=thr)
        if r["n_candidates"] == 0 and thr > 0.35:
            self.board.decide(f"no candidates with consensus >= {thr} -> lowering threshold to 0.35")
            self.board.fact("min_prob", 0.35)
            r = self.call("consensus", "build", min_prob=0.35)
        return r

    def s_fit(self):
        thr = self.board.fact("min_prob") or CFG.min_prob
        return self.call("fitting", "fit", top=CFG.n_top, min_prob=thr, steps=CFG.fit_steps,
                         max_steps=CFG.fit_max_steps, n_jobs=min(4, os.cpu_count() or 2))

    def s_report(self):
        return self.call("reporting", "report", n_jobs=min(4, os.cpu_count() or 2))

    def s_explain(self):
        r = self.call("explainer", "explain", top=CFG.llm_top, use_llm=CFG.use_llm, llm_id=CFG.llm_id)
        if r["mode"] == "llm":
            self.call("explainer", "release")
        return r

    def s_active_learning(self):
        q = self.call("active_learning", "queue", k=25)
        s = self.call("active_learning", "simulate", rounds=CFG.al_rounds, batch=CFG.al_batch, seeds=CFG.al_seeds)
        return {"queue": q, "simulation": s}

    STAGES = ["inventory", "preprocess", "detect", "features", "train_tabular", "train_cnn", "consensus",
              "fit", "report", "explain", "active_learning"]

    def _stage(self, name, force):
        if self.board.done(name) and name not in force and "all" not in force:
            print(f"\n=== {name}: already done (resumed) ===")
            return
        print(f"\n=== STAGE: {name} ===")
        self.ask("monitor", "snapshot", stage=name, phase="start")
        t0 = time.time()
        try:
            info = getattr(self, "s_" + name)()
            self.board.mark(name, "done", info, round(time.time() - t0, 1))
        except Exception as e:                                               # noqa: BLE001
            self.board.mark(name, "failed", str(e)[:600], round(time.time() - t0, 1))
            print(f"  stage {name} FAILED: {str(e).splitlines()[0][:200]}")
            if name not in self.OPTIONAL:
                raise
            self.board.decide(f"optional stage '{name}' failed; continuing")
        finally:
            self.ask("monitor", "snapshot", stage=name, phase="end")

    def run(self, force=(), stages=None):
        force = set(force)
        print(f"Run '{RUN_ID}' | smoke={CFG.smoke} | outputs -> {OUT}")
        for name in (stages or self.STAGES):
            self._stage(name, force)
        return self.summary()

    def summary(self):
        rows = [{"stage": k, "status": v["status"], "seconds": v["seconds"]} for k, v in self.board.d["stages"].items()]
        return pd.DataFrame(rows)

    # ------------------------------------------------- human-in-the-loop update
    def learn_from_experts(self, labelled_queue_csv):
        """Ingest expert labels, retrain the classifier, refresh consensus. The registry keeps the better model."""
        n = self.call("active_learning", "ingest", path=str(labelled_queue_csv))
        self.board.decide(f"{n['labels_total']} expert labels ingested -> retraining classifier")
        r = self.call("classification", "train", trials=CFG.optuna_trials, seed=CFG.seed, note="retrain with expert labels")
        c = self.call("consensus", "build", min_prob=self.board.fact("min_prob") or CFG.min_prob)
        return {"retrain": r, "consensus": c}

print("orchestrator defined")

## 11. Launch the platform

In [ ]:
bus   = MessageBus(RUN_DIR / "messages.jsonl")
board = Blackboard(RUN_DIR / "blackboard.json")
agents = {a.name: a for a in (
    MonitorAgent(bus, board), AcquisitionAgent(bus, board), PreprocessingAgent(bus, board), DetectionAgent(bus, board),
    FeatureAgent(bus, board), ClassificationAgent(bus, board), DeepVettingAgent(bus, board), ConsensusAgent(bus, board),
    ActiveLearningAgent(bus, board), FittingAgent(bus, board), ReportingAgent(bus, board), ExplainerAgent(bus, board))}
orch = Orchestrator(bus, board)
print("registered agents:", ", ".join(bus.agents))

In [ ]:
# Runs every stage; already-finished stages are skipped, so this cell is safe to re-run after an interruption.
# To redo a stage: orch.run(force=("fit", "report")); to redo everything: orch.run(force=("all",))
timeline = orch.run()
timeline

## 12. Results

In [ ]:
# What the agents said to each other (audit log) and how long each stage took
log = pd.read_json(RUN_DIR / "messages.jsonl", lines=True)
log = log[~log["action"].str.endswith(".result")][["sender", "recipient", "action", "status", "seconds"]]
display(log.tail(25))
display(pd.DataFrame(orch.ask("monitor", "summary").payload["rows"]))
print("\nDecisions taken by the orchestrator:")
for d in board.d["decisions"]:
    print(" -", d["text"])

In [ ]:
# Model registry (tabular) and head-to-head of the three opinions on the same held-out stars
reg = pd.DataFrame(registry())
if len(reg):
    display(reg[["version", "champion", "pr_auc", "roc_auc", "macro_f1", "brier", "xgb_device", "trials", "smote", "human_overrides"]].round(3))
m = json.loads((OUT / "consensus_metrics.json").read_text())
display(pd.DataFrame({k: v for k, v in m.items() if v}).T.round(3))

In [ ]:
# Ranked candidate catalogue
cat = pd.read_csv(OUT / "candidate_catalog.csv")
cols = [c for c in ["rank", "tic_id", "verdict", "rank_score", "p_transit", "p_tab", "p_cnn", "disagreement", "period_d", "depth_ppm",
                    "rp_rearth", "size_class", "snr", "teq_k", "in_hz", "split"] if c in cat]
display(cat[cols].head(15).round(3))

In [ ]:
# Diagnostics for the top 2 candidates + plain-English summaries
from IPython.display import Image, display, Markdown
for tic in cat["tic_id"].head(2):
    for name in ("1_lightcurve", "3_fold", "5_odd_even"):
        p = OUT / "plots" / tic / f"{name}.png"
        if p.exists():
            display(Markdown(f"**{tic}: {name}**")); display(Image(str(p), width=620))
    print("PDF report:", OUT / "reports" / f"{tic}.pdf")
display(Markdown((OUT / "candidate_summaries.md").read_text()))

## 13. Optional: ask the platform questions (GPU LLM)

Loads the LLM once and answers questions about the ranked catalogue. It only sees the table, so it cannot invent numbers.

In [ ]:
if CFG.use_llm:
    for q in ["Which candidates passed all vetting checks and have SNR above 10?",
              "Which candidate is most likely inside the habitable zone, and how sure are we?"]:
        print("Q:", q)
        print("A:", orch.ask("explainer", "ask", question=q, llm_id=CFG.llm_id).payload.get("answer", "(LLM unavailable)"), "\n")
    orch.ask("explainer", "release")
else:
    print("USE_LLM is False; set it to True in the settings cell to enable this.")

## 14. Optional: close the human-in-the-loop

1. Download `outputs/review_queue.csv`, fill the `human_label` column with `transit` or `other`, and upload it back (or edit it in place in the Kaggle file browser).
2. Run the cell below. The orchestrator ingests the labels, retrains the classifier (the registry promotes the new version only if it is not worse) and refreshes the consensus scores.

In [ ]:
# queue = OUT / "review_queue.csv"      # after you have filled in human_label
# print(orch.learn_from_experts(queue))

## 15. Package everything for download

In [ ]:
import zipfile
zpath = Path("/kaggle/working") / f"lunavision_{RUN_ID}_results.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for root in (OUT, MODELS, RUN_DIR):
        for f in Path(root).rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(WS))
print(zpath, f"{zpath.stat().st_size / 1e6:.1f} MB")

## Troubleshooting

* **`No src/ folder` at setup:** attach the dataset containing the repo `src/` (with `fitting.py`) via *Add Input*.
* **`no light curves and no feature matrix`:** attach `lc_detrended/` (fastest) or `lc_raw/` + `lc_injected/`; also attach the catalog CSVs.
* **CNN stage says too few valid views:** the feature matrix has no `period/t0` rows, or `lc_detrended/` is not attached (the CNN reads the light curves).
* **A stage failed:** fix the cause, then rerun the launch cell. Finished stages are skipped; use `orch.run(force=("stage",))` to redo one.
* **CuPy import error:** harmless: feature extraction falls back to CPU (slower). Make sure the accelerator is set to GPU.
* **LLM download fails (no internet):** set `USE_LLM = False`; the explainer then writes template summaries.
* **Run out of GPU memory:** lower `cnn_max_rows`, or use a smaller `LLM_ID` such as `Qwen/Qwen2.5-1.5B-Instruct`.